# EGN ablation study — five datasets, eight variantsEach dataset gets its **own loader cell** and its **own ablation cell**, so a failure in onedataset does not cost you the others. Results are appended to `/kaggle/working/egn_results.json`after every single run, which means a session timeout loses at most one run and re-executing acell resumes where it stopped.The last cell assembles the LaTeX table.**Before you start*** `egnlib >= 0.2.0` is required — 0.1.0 has no `pool="arithmetic"`, no `constrained=False`,  no `separation`, and no `head="logeig"`, so four of the eight rows cannot be run against it.* Every configuration keeps **more than one manifold channel at the pooling layer**. With a  single channel the pool is the identity and the two Karcher rows would silently equal the  full model. This is the easiest way to produce a meaningless ablation table, so each dataset  block states its channel count explicitly.* Set `N_SEEDS = 3` for the numbers you actually publish. The default of 1 is for the first  pass, when you want to know whether the pipeline runs at all.

In [ ]:
# egnlib >= 0.2.0. Pick whichever of these applies to you:#   * published to PyPI      -> the line below just works#   * not yet published      -> upload the wheel as a Kaggle Dataset and use the fallback#   * private GitHub repo    -> pip install git+https://<token>@github.com/kraihan/EGN.git!pip install -q "egnlib>=0.2.0" || pip install -q /kaggle/input/egnlib-wheel/egnlib-0.2.0-py3-none-any.whlimport egnassert egn.__version__ >= "0.2.0", f"need egnlib>=0.2.0, found {egn.__version__}"print("egnlib", egn.__version__)

## Shared setup`VARIANTS` is the table: one entry per row, expressed as an override of the full model. Nothingbelow this cell knows which dataset it is running on.

In [ ]:
import json, os, re, time, warningsimport numpy as npimport torchfrom egn import EGNClassifierwarnings.filterwarnings("ignore", category=UserWarning)RESULTS_PATH = "/kaggle/working/egn_results.json"N_SEEDS = 1          # raise to 3 for the published tableSEEDS = [1024, 2048, 4096][:N_SEEDS]def pick_device() -> str:    """A CUDA device that can actually run kernels from this PyTorch build."""    if torch.cuda.is_available():        try:            torch.zeros(1, device="cuda").add_(1).cpu()            print("GPU:", torch.cuda.get_device_name(0),                  f"({torch.cuda.device_count()} visible)")            return "cuda"        except Exception as exc:            print(f"CUDA present but unusable ({type(exc).__name__}) -> CPU")    return "cpu"DEVICE = pick_device()# ---------------------------------------------------------------- the ablation# Row order matches the paper table.FULL = dict(    head="geodesic",      # geodesic prototype head    pool="frechet",       # Karcher (Frechet) mean pooling    bias=True,            # geometric bias    dropout=0.1,          # geodesic soft dropout    separation=0.01,      # prototype repulsion    constrained=True,     # Stiefel-constrained bilinear maps)VARIANTS = {    "EGN (full)":                    {},    "Geodesic head -> LogEig+FC":    dict(head="logeig"),    "Karcher -> log-Euclidean":      dict(pool="logeuclid"),    "Karcher -> arithmetic mean":    dict(pool="arithmetic"),    "w/o geometric bias":            dict(bias=False),    "w/o geodesic soft dropout":     dict(dropout=0.0),    "w/o prototype repulsion":       dict(separation=0.0),    "Stiefel -> unconstrained":      dict(constrained=False),}# ------------------------------------------------------------------- storagedef load_results():    if os.path.exists(RESULTS_PATH):        with open(RESULTS_PATH) as fh:            return json.load(fh)    return {}def save_result(dataset, variant, seed, accuracy, seconds):    r = load_results()    r.setdefault(dataset, {}).setdefault(variant, {})[str(seed)] = {        "accuracy": float(accuracy), "seconds": float(seconds)    }    with open(RESULTS_PATH, "w") as fh:        json.dump(r, fh, indent=1)# --------------------------------------------------------------------- splitsdef stratified_split(y, train_fraction=0.7, seed=0, groups=None):    """Stratified by class, or grouped by subject when ``groups`` is given.    Subject-independent splitting is the honest protocol for EEG: a within-subject    split leaks identity through the covariance structure and inflates every row of    the table by roughly the same amount, which hides exactly the differences an    ablation is supposed to expose.    """    rng = np.random.default_rng(seed)    if groups is not None:        uniq = np.unique(groups)        rng.shuffle(uniq)        cut = max(int(round(len(uniq) * train_fraction)), 1)        train_g = set(uniq[:cut].tolist())        mask = np.array([g in train_g for g in groups])        return np.flatnonzero(mask), np.flatnonzero(~mask)    tr, te = [], []    for c in np.unique(y):        idx = np.flatnonzero(y == c)        rng.shuffle(idx)        cut = max(int(round(len(idx) * train_fraction)), 1)        tr.append(idx[:cut]); te.append(idx[cut:])    return np.concatenate(tr), np.concatenate(te)# ------------------------------------------------------------------ the runnerdef run_ablation(dataset, X, y, base, groups=None, train_fraction=0.7, variants=None,                 skip_done=True):    """Fit every variant on one dataset and record the test accuracy."""    variants = variants or VARIANTS    done = load_results().get(dataset, {})    print(f"\n{'=' * 78}\n{dataset}: X {tuple(X.shape)}  y {tuple(y.shape)}  "          f"{len(np.unique(y))} classes\n{'=' * 78}")    for name, override in variants.items():        for seed in SEEDS:            if skip_done and str(seed) in done.get(name, {}):                print(f"  {name:<30} seed {seed}  cached "                      f"{done[name][str(seed)]['accuracy'] * 100:5.2f}")                continue            cfg = {**FULL, **base, **override}            tr, te = stratified_split(y, train_fraction, seed, groups)            t0 = time.perf_counter()            try:                clf = EGNClassifier(device=DEVICE, seed=seed, verbose=0, **cfg)                clf.fit(X[tr], y[tr])                acc = clf.score(X[te], y[te])            except Exception as exc:                print(f"  {name:<30} seed {seed}  FAILED: {type(exc).__name__}: {exc}")                continue            dt = time.perf_counter() - t0            save_result(dataset, name, seed, acc, dt)            print(f"  {name:<30} seed {seed}  acc {acc * 100:5.2f}   ({dt:5.1f}s)")    return load_results()[dataset]

## 1. Radar3000 complex signals of length 1000, three classes. The signal is cut into overlapping windows(length 20, hop 10, giving 99 windows) and each complex window is embedded as its real/imaginarystack, so a 20-sample complex window becomes a 40-dimensional real vector. The covariance ofthose vectors is the SPD descriptor.The embedding matters: a complex covariance is Hermitian, not symmetric, and the real embedding`[Re; Im]` is the standard isometry that turns a Hermitian problem into a real SPD one of twicethe size.

In [ ]:
RADAR_DIR = "/kaggle/working/data/radar_npy/radar"def parse_label(name):    m = re.search(r"_(\d+)\.npy$", name) or re.search(r"(\d+)\.npy$", name)    return int(m.group(1)) if m else Nonedef load_radar(root=RADAR_DIR, window=20, hop=10):    files = sorted(f for f in os.listdir(root) if f.endswith(".npy"))    labels = [parse_label(f) for f in files]    if any(l is None for l in labels):        raise ValueError(f"could not read a label from e.g. {files[:3]}")    X = []    for f in files:        z = np.load(os.path.join(root, f)).ravel()        starts = range(0, len(z) - window + 1, hop)        W = np.stack([z[s:s + window] for s in starts], axis=1)      # (window, n_windows)        X.append(np.concatenate([W.real, W.imag], axis=0))            # (2*window, n_windows)    X = np.stack(X).astype(np.float32)    return X, np.asarray(labels)X_radar, y_radar = load_radar()print("radar", X_radar.shape, "labels", np.unique(y_radar, return_counts=True))

In [ ]:
RADAR_BASE = dict(    input_kind="signal",   # (N, 40, 99): the covariance is formed inside the model    branches=4,            # 4 manifold channels -> the pooling ablations are meaningful    channels=4,    dims=[40, 24, 12],    epochs=60, batch_size=64, lr=5e-3, ridge=1e-3, scheduler="cosine",)run_ablation("Radar", X_radar, y_radar, RADAR_BASE, train_fraction=0.7)

## 2. HDM052086 skeleton sequences, 117 action classes, already supplied as 93×93 covariance descriptors.The width schedule 93→50→30 is the one SPDNet uses on this dataset, so the numbers staycomparable to the published baselines.`channels=4` is what gives the pooling layer something to pool: the input carries a singlematrix per sample, and the first `BiMap` expands it to four manifold channels.

In [ ]:
HDM05_DIR = "/kaggle/working/data/hdm05_raw/HDM05"def load_hdm05(root=HDM05_DIR, cache="/kaggle/working/hdm05.npz"):    if cache and os.path.exists(cache):        b = np.load(cache); return b["X"], b["y"]    files = sorted(f for f in os.listdir(root) if f.endswith(".npy"))    X = np.stack([np.squeeze(np.load(os.path.join(root, f))) for f in files]).astype(np.float32)    y = np.array([parse_label(f) for f in files])    if cache:        np.savez_compressed(cache, X=X, y=y)    return X, yX_hdm, y_hdm = load_hdm05()eig = np.linalg.eigvalsh(X_hdm[:64])print("hdm05", X_hdm.shape, len(np.unique(y_hdm)), "classes",      f"| min eig {eig.min():.2e} | cond {eig.max() / max(eig.min(), 1e-12):.1e}")

In [ ]:
HDM05_BASE = dict(    input_kind="spd",    branches=1,    channels=4,            # the stem expands 1 -> 4 channels, so pooling is not a no-op    dims=[93, 50, 30],    epochs=80, batch_size=30, lr=1e-2, ridge=1e-3, scheduler="cosine",)run_ablation("HDM05", X_hdm, y_hdm, HDM05_BASE, train_fraction=0.5)

## 3. Psychiatric (schizophrenia vs healthy control)212 subjects, binary. The rows are feature vectors, not matrices, so the SPD structure has tocome from somewhere: the `COH.*` columns are pairwise coherences between the 19 electrodes ineach of the 6 bands, which reassemble exactly into **six 19×19 connectivity matrices persubject** — a natural six-channel manifold input.Coherence matrices are symmetric with a unit diagonal but need not be positive definite afterestimation, hence the ridge. If the column-name parsing fails on your copy of the CSV, thefallback path reshapes the numeric features into a channel × feature matrix and takes itscovariance; it is a weaker descriptor, and the cell tells you which path it used.

In [ ]:
import pandas as pdPSY_CSV = "/kaggle/input/eeg-psychiatric-disorders-dataset/EEG.machinelearing_data_BRMH.csv"def load_psychiatric(path=PSY_CSV):    data = pd.read_csv(path)    sub = data[data["main.disorder"].isin(["Healthy control", "Schizophrenia"])].copy()    y = (sub["main.disorder"] == "Schizophrenia").astype(int).values    coh = [c for c in sub.columns if c.startswith("COH.")]    parsed, bands, chans = {}, [], []    for c in coh:        parts = c.split(".")        if len(parts) == 7:            band, a, b = parts[2], parts[4], parts[6]            parsed[c] = (band, a, b)            bands.append(band); chans += [a, b]    if parsed:        bands = sorted(set(bands)); chans = sorted(set(chans))        bi = {b: i for i, b in enumerate(bands)}; ci = {c: i for i, c in enumerate(chans)}        n, K = len(chans), len(bands)        X = np.tile(np.eye(n, dtype=np.float32), (len(sub), K, 1, 1))        for col, (band, a, b) in parsed.items():            v = sub[col].astype(np.float32).values            X[:, bi[band], ci[a], ci[b]] = v            X[:, bi[band], ci[b], ci[a]] = v        X = np.nan_to_num(X, nan=0.0)        print(f"COH path: {K} bands x {n} channels -> {X.shape}")        return X, y, "spd"    num = sub.select_dtypes(include=[np.number]).drop(columns=["no."], errors="ignore")    V = np.nan_to_num(num.values.astype(np.float32))    V = (V - V.mean(0)) / (V.std(0) + 1e-6)    width = 19    V = V[:, : (V.shape[1] // width) * width].reshape(len(V), width, -1)    print(f"fallback path: reshaped features -> {V.shape}")    return V, y, "signal"X_psy, y_psy, PSY_KIND = load_psychiatric()print("psychiatric", X_psy.shape, np.bincount(y_psy))

In [ ]:
PSY_BASE = dict(    input_kind=PSY_KIND,    branches=1 if PSY_KIND == "spd" else 4,    channels=6,    dims=[X_psy.shape[-1], 12, 8] if PSY_KIND == "spd" else [19, 12, 8],    epochs=120, batch_size=16, lr=5e-3, ridge=1e-2, dropout=0.1, scheduler="cosine",)# 212 subjects is small enough that one split is mostly noise -- use every seed you can affordrun_ablation("Psychiatric", X_psy, y_psy, PSY_BASE, train_fraction=0.7)

## 4. DEAP1280 trials (32 participants × 40 videos) of differential-entropy features, binary valence.Each trial is `(1024, 60)`; the 1024 rows are read as 32 EEG channels × 32 sub-features, so atrial becomes a `(32, 32 × 60)` multichannel signal and the model forms a 32×32 channelcovariance.**Check `DEAP_CHANNELS` against your own feature extractor** before you believe these numbers.If the 1024 axis is not channel-major, the covariance is over the wrong quantity and the wholecolumn is meaningless — the reshape is stated as an assumption for exactly that reason.The split is by participant. A within-participant split on DEAP is the single most common wayto report inflated emotion-recognition accuracy.

In [ ]:
DEAP_DIR = "/kaggle/input/deap-research"DEAP_CHANNELS = 32          # <- verify against your own DE extractorFRAMES = 60def load_deap(root=DEAP_DIR, cache="/kaggle/working/deap.npz"):    if cache and os.path.exists(cache):        b = np.load(cache); return b["X"], b["y"], b["g"]    trials = []    for i in range(1, 33):        for j in range(1, 41):            df = pd.read_csv(f"{root}/DE/participant{i}video{j}.txt", header=None,                             usecols=list(range(FRAMES)), delimiter=",")            trials.append(df.values)    X = np.asarray(trials, dtype=np.float32)                       # (1280, 1024, 60)    lo = X.min(axis=(1, 2), keepdims=True); hi = X.max(axis=(1, 2), keepdims=True)    X = (X - lo) / (hi - lo + 1e-8)    X = X.reshape(len(X), DEAP_CHANNELS, -1)                       # (1280, 32, 32*60)    lab = pd.read_csv(f"{root}/label.txt", header=None, delimiter=",",                      usecols=list(range(4)))    lab.columns = ["valence", "arousal", "dominance", "liking"]    y = (lab["valence"].values >= 5).astype(int)                   # valence, binarised    g = np.repeat(np.arange(32), 40)                               # participant id    if cache:        np.savez_compressed(cache, X=X, y=y, g=g)    return X, y, gX_deap, y_deap, g_deap = load_deap()print("deap", X_deap.shape, np.bincount(y_deap), f"{len(np.unique(g_deap))} participants")

In [ ]:
DEAP_BASE = dict(    input_kind="signal",    branches=4,    channels=4,    dims=[DEAP_CHANNELS, 20, 12],    epochs=60, batch_size=64, lr=5e-3, ridge=1e-3, scheduler="cosine",)run_ablation("DEAP", X_deap, y_deap, DEAP_BASE, groups=g_deap, train_fraction=0.75)

## 5. SEED675 trials (15 subjects × 45 trials) of DE-LDS features, shape `(62 channels, 185 windows,5 bands)`, three emotion classes. One 62×62 channel covariance per frequency band gives afive-channel manifold input, which keeps the bands separate instead of averaging them into asingle covariance.Load the file **without** the shuffle from your original pipeline: the subject grouping ispositional, and shuffling first destroys the information the subject-independent split needs.

In [ ]:
SEED_NPY = "/kaggle/input/de-lds-mat-seed-dataset-de-features/SEED_DE_LDS_label.npy"def load_seed(path=SEED_NPY, cache="/kaggle/working/seed.npz"):    if cache and os.path.exists(cache):        b = np.load(cache); return b["X"], b["y"], b["g"]    blob = np.load(path, allow_pickle=True).item()    data = np.asarray(blob["data"], dtype=np.float32)     # (675, 62, windows, 5)    y = np.asarray(blob["labels"]).reshape(-1)    # per-band channel covariance: (N, 5, 62, 62)    Z = np.transpose(data, (0, 3, 1, 2))                  # (N, bands, channels, windows)    Z = Z - Z.mean(-1, keepdims=True)    X = (Z @ np.swapaxes(Z, -1, -2)) / max(Z.shape[-1] - 1, 1)    X = X.astype(np.float32)    per_subject = len(y) // 15    g = np.repeat(np.arange(15), per_subject)[: len(y)]    if cache:        np.savez_compressed(cache, X=X, y=y, g=g)    return X, y, gX_seed, y_seed, g_seed = load_seed()print("seed", X_seed.shape, "labels", np.unique(y_seed, return_counts=True),      f"| {len(np.unique(g_seed))} subjects")

In [ ]:
SEED_BASE = dict(    input_kind="spd",      # (675, 5, 62, 62): five bands = five manifold channels    branches=1,    channels=5,    dims=[62, 32, 16],    epochs=80, batch_size=32, lr=5e-3, ridge=1e-3, scheduler="cosine",)run_ablation("SEED", X_seed, y_seed, SEED_BASE, groups=g_seed, train_fraction=0.8)

## The tableReads whatever is in `egn_results.json`, so it works on a partial run — missing cells show as`--` rather than failing.

In [ ]:
DATASETS = ["Radar", "HDM05", "Psychiatric", "DEAP", "SEED"]ROWS = list(VARIANTS)results = load_results()def cell(dataset, variant):    runs = results.get(dataset, {}).get(variant, {})    if not runs:        return None, None    acc = np.array([r["accuracy"] for r in runs.values()]) * 100    return acc.mean(), (acc.std() if len(acc) > 1 else None)table = pd.DataFrame(index=ROWS, columns=DATASETS, dtype=object)for v in ROWS:    for d in DATASETS:        m, s = cell(d, v)        table.loc[v, d] = "--" if m is None else (f"{m:.2f}" if s is None else f"{m:.2f}±{s:.2f}")display(table)print("\ntotal GPU time:",      f"{sum(r['seconds'] for ds in results.values() for v in ds.values() for r in v.values()) / 60:.1f} min")

In [ ]:
def latex_table(table):    head = (r"\begin{tabular}{lccccc}" "\n" r"\toprule" "\n"            r"\textbf{Variant} & " +            " & ".join(rf"\textbf{{{d}}}" for d in DATASETS) + r" \\" "\n" r"\midrule")    lines = [head]    for i, v in enumerate(ROWS):        label = v.replace("->", r"$\to$").replace("w/o", "w/o")        vals = [str(table.loc[v, d]) for d in DATASETS]        if i == 0:                       # bold the full model            vals = [rf"\textbf{{{x}}}" for x in vals]            label = label + " " * max(0, 32 - len(label))        lines.append(f"{label:<34}& " + " & ".join(vals) + r" \\")    lines += [r"\bottomrule", r"\end{tabular}"]    return "\n".join(lines)latex = latex_table(table)print(latex)with open("/kaggle/working/ablation_table.tex", "w") as fh:    fh.write(latex)